# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading, exploring, and processing a dataset using the `mlcroissant` library. We will demonstrate how to access record sets, fields, and columns by their `@id`, and perform exploratory data analysis.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset title: {metadata.name}")
print(f"Dataset description: {metadata.description}")
print(f"Dataset identifier: {metadata.identifier}")
print(f"Dataset version: {metadata.version}")


## 2. Data Overview

Review available record sets, fields, and their IDs. We will list all record sets, their `@id`, and their respective fields.

In [ ]:
# List all available record sets by their @id
record_sets = dataset.record_sets
print("Available Record Sets:")
for rs in record_sets:
    print(f"- RecordSet @id: {rs.id}, name: {rs.name if hasattr(rs, 'name') else '[no name]'}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    Field @id: {field.id}, name: {field.name if hasattr(field, 'name') else '[no name]'}")
    print()


## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data from each record set by @id
dataframes = {}
record_set_ids = [rs.id for rs in record_sets]
print("Record set @ids for extraction:", record_set_ids)

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        dataframes[record_set_id] = pd.DataFrame(records)

# Display columns for each DataFrame
for record_set_id, df in dataframes.items():
    print(f"\nColumns in record set {record_set_id}:")
    print(df.columns.tolist())
    print(df.head())


## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps: filtering numeric fields, normalization, and grouping. Reference all fields by their `@id`. Adjust the code based on available numeric fields and group fields discovered in the overview.

In [ ]:
# Choose a record set and identify numeric fields
# We'll select the first record set with data
if len(dataframes) > 0:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    print(f"Working with record set @id: {record_set_id}")

    # Identify numeric fields by @id
    numeric_fields = [col for col in df.columns if df[col].dtype in ['float64', 'int64']]
    print(f"Numeric fields: {numeric_fields}")
    if numeric_fields:
        numeric_field_id = numeric_fields[0]  # Just use first numeric field

        # Filter records for numeric_field_id > threshold
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by a categorical field
        cat_fields = [col for col in df.columns if df[col].dtype == 'object']
        if cat_fields:
            group_field_id = cat_fields[0]
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"Grouped {numeric_field_id} by {group_field_id}:")
            print(grouped_df.head())
        else:
            print("No categorical fields found to group by.")
    else:
        print("No numeric fields found in the record set.")
else:
    print("No dataframes available for analysis.")


## 5. Visualization

Visualize data distributions or relationships between fields. For demonstration, we plot a histogram of the chosen numeric field and a bar chart of group means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization only if analysis yielded numeric field
if len(dataframes) > 0 and numeric_fields:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    if cat_fields:
        # Bar plot for mean by group
        mean_by_group = df.groupby(group_field_id)[numeric_field_id].mean()
        mean_by_group = mean_by_group.dropna()
        plt.figure(figsize=(8, 4))
        mean_by_group.plot(kind="bar")
        plt.title(f"Mean {numeric_field_id} grouped by {group_field_id}")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xlabel(group_field_id)
        plt.show()


## 6. Conclusion

In this notebook, we loaded the FAIR² dataset using the `mlcroissant` library, explored the record sets and fields by their `@id`s, and performed basic data extraction and analysis. You can extend this workflow to deeper analyses, including missing data handling and advanced visualizations, using the rich Croissant schema information and `mlcroissant` tools for reproducible and structured FAIR dataset exploration.